# Author Embeddings Upload to Vector Database

This notebook loads author and works data from Azure Blob Storage and creates author embeddings based on their paper abstracts. The embeddings are uploaded to a local Milvus vector database using the `/authors/vectors` endpoint.

## Overview
1. Connect to Azure Blob Storage
2. Load author and works data
3. Group works by author to collect abstracts
4. Upload author embeddings to the vector database

## 1. Install and Import Dependencies

In [3]:
# Import required libraries
import os
import json
import requests
from collections import defaultdict
from typing import Dict, List, Optional
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm

In [4]:
import numpy as np
import multiprocessing
from sentence_transformers import SentenceTransformer
from concurrent.futures import ProcessPoolExecutor, as_completed

## 2. Configure Azure Storage and Vector DB Connection

In [16]:
# Azure Storage Configuration
AZURE_STORAGE_CONNECTION_STRING = os.environ.get('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = 'clean'  # Container with cleaned data
AUTHORS_PREFIX = 'dtic/authors/'
WORKS_PREFIX = 'dtic/works/'

# Vector DB Configuration
VECTOR_DB_BASE_URL = 'http://localhost:8002'
VECTOR_UPLOAD_ENDPOINT = f'{VECTOR_DB_BASE_URL}/authors/vector'

# Processing Configuration
NUM_WORKERS = 2  # Number of parallel processes (start with 2, increase if stable)
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # Should match vector-db service
BATCH_SIZE = 100  # Process embeddings in batches to manage memory
CACHE_FILE = 'author_abstracts_cache.json'  # Cache file for author-abstracts mapping

# Note: If you encounter BrokenProcessPool errors on Windows, try:
# 1. Reducing NUM_WORKERS to 1 or 2
# 2. Restarting the kernel before running the processing cell
# 3. Running from a Python script instead of Jupyter if issues persist

# Verify connection string
if not AZURE_STORAGE_CONNECTION_STRING:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING environment variable not set!")

print("✓ Configuration loaded")
print(f"  Container: {CONTAINER_NAME}")
print(f"  Authors prefix: {AUTHORS_PREFIX}")
print(f"  Works prefix: {WORKS_PREFIX}")
print(f"  Vector DB: {VECTOR_DB_BASE_URL}")
print(f"  Workers: {NUM_WORKERS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Cache file: {CACHE_FILE}")

✓ Configuration loaded
  Container: clean
  Authors prefix: dtic/authors/
  Works prefix: dtic/works/
  Vector DB: http://localhost:8002
  Workers: 2
  Batch size: 100
  Cache file: author_abstracts_cache.json


## 3. Check for Cached Data

**Important:** If the cache file exists, you can skip loading from Azure Blob Storage (cells 4, 5, 6) and jump directly to cell with the cache loading.

In [6]:
# Check if cache file exists
if os.path.exists(CACHE_FILE):
    print(f"✓ Cache file found: {CACHE_FILE}")
    print(f"  You can skip the blob storage loading (cells 4, 5, 6) and jump to the cache loading cell.")
    print(f"  Or continue to reload fresh data from Azure Blob Storage.")
else:
    print(f"✗ No cache file found at '{CACHE_FILE}'")
    print(f"  You must load data from Azure Blob Storage (continue with cells below).")

✗ No cache file found at 'author_abstracts_cache.json'
  You must load data from Azure Blob Storage (continue with cells below).


## 4. Connect to Azure Blob Storage

In [7]:
# Initialize Blob Service Client (skip if loading from cache)
if os.path.exists(CACHE_FILE):
    print(f"⊙ Cache file exists ({CACHE_FILE}), skipping blob storage connection.")
    print("  Jump to cell 8 to load from cache.")
else:
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    print("✓ Connected to Azure Blob Storage")

✓ Connected to Azure Blob Storage


## 5. Load Authors from Blob Storage

**Note:** Skip this cell if you're loading from cache.

In [8]:
if os.path.exists(CACHE_FILE):
    print(f"⊙ Cache file exists ({CACHE_FILE}), skipping author loading from blob storage.")
    print("  Jump to cell 8 to load from cache.")
else:
    def load_authors_from_blob(container_client, prefix: str, max_authors: Optional[int] = None) -> Dict[str, dict]:
        """
        Load author data from Azure Blob Storage.
        
        Args:
            container_client: Azure container client
            prefix: Blob prefix for authors (e.g., 'dtic/authors/')
            max_authors: Maximum number of authors to load (None = all)
            
        Returns:
            Dictionary mapping author_id to author data
        """
        authors = {}
        blob_list = container_client.list_blobs(name_starts_with=prefix)
        
        print(f"Loading authors from {prefix}...")
        
        for i, blob in enumerate(tqdm(blob_list)):
            if max_authors and i >= max_authors:
                break
                
            # Download blob content
            blob_client = container_client.get_blob_client(blob.name)
            blob_data = blob_client.download_blob().readall()
            
            try:
                author_data = json.loads(blob_data)
                author_id = author_data.get('id')
                if author_id:
                    authors[author_id] = author_data
            except json.JSONDecodeError as e:
                print(f"Error decoding {blob.name}: {e}")
                continue
        
        print(f"✓ Loaded {len(authors)} authors")
        return authors
    
    # Load authors (set max_authors to limit for testing, None for all)
    authors = load_authors_from_blob(container_client, AUTHORS_PREFIX, max_authors=None)
    print(f"\nTotal authors loaded: {len(authors)}")

Loading authors from dtic/authors/...


94367it [25:13, 62.37it/s]

✓ Loaded 94367 authors

Total authors loaded: 94367


## 6. Load Works from Blob Storage

**Note:** Skip this cell if you're loading from cache.

In [9]:
if os.path.exists(CACHE_FILE):
    print(f"⊙ Cache file exists ({CACHE_FILE}), skipping works loading from blob storage.")
    print("  Jump to cell 8 to load from cache.")
else:
    def load_works_from_blob(container_client, prefix: str, max_works: Optional[int] = None) -> List[dict]:
        """
        Load works data from Azure Blob Storage.
        
        Args:
            container_client: Azure container client
            prefix: Blob prefix for works (e.g., 'dtic/works/')
            max_works: Maximum number of works to load (None = all)
            
        Returns:
            List of work dictionaries
        """
        works = []
        blob_list = container_client.list_blobs(name_starts_with=prefix)
        
        print(f"Loading works from {prefix}...")
        
        for i, blob in enumerate(tqdm(blob_list)):
            if max_works and i >= max_works:
                break
                
            # Download blob content
            blob_client = container_client.get_blob_client(blob.name)
            blob_data = blob_client.download_blob().readall()
            
            try:
                work_data = json.loads(blob_data)
                if work_data.get('abstract'):  # Only include works with abstracts
                    works.append(work_data)
            except json.JSONDecodeError as e:
                print(f"Error decoding {blob.name}: {e}")
                continue
        
        print(f"✓ Loaded {len(works)} works with abstracts")
        return works
    
    # Load works (set max_works to limit for testing, None for all)
    works = load_works_from_blob(container_client, WORKS_PREFIX, max_works=None)
    print(f"\nTotal works with abstracts: {len(works)}")

Loading works from dtic/works/...


28071it [06:36, 70.73it/s]

✓ Loaded 27124 works with abstracts

Total works with abstracts: 27124


## 7. Group Works by Author

**Note:** Skip this cell if you're loading from cache.

In [10]:
if os.path.exists(CACHE_FILE):
    print(f"⊙ Cache file exists ({CACHE_FILE}), skipping grouping and cache creation.")
    print("  Jump to cell 8 to load from cache.")
else:
    def group_abstracts_by_author(works: List[dict]) -> Dict[str, List[str]]:
        """
        Group abstracts by author ID.
        
        Args:
            works: List of work dictionaries
            
        Returns:
            Dictionary mapping author_id to list of abstracts
        """
        author_abstracts = defaultdict(list)
        
        print("Grouping abstracts by author...")
        for work in tqdm(works):
            abstract = work.get('abstract', '').strip()
            if not abstract:
                continue
                
            # Each work can have multiple authors
            authors_list = work.get('authors', [])
            for author_entry in authors_list:
                author_id = author_entry.get('author_id')
                if author_id:
                    author_abstracts[author_id].append(abstract)
        
        print(f"✓ Grouped abstracts for {len(author_abstracts)} authors")
        return dict(author_abstracts)
    
    # Group abstracts
    author_abstracts = group_abstracts_by_author(works)
    
    # Show statistics
    abstracts_per_author = [len(abstracts) for abstracts in author_abstracts.values()]
    print(f"\nStatistics:")
    print(f"  Authors with abstracts: {len(author_abstracts)}")
    print(f"  Total abstract-author pairs: {sum(abstracts_per_author)}")
    print(f"  Average abstracts per author: {sum(abstracts_per_author) / len(abstracts_per_author):.2f}")
    print(f"  Max abstracts for one author: {max(abstracts_per_author)}")
    print(f"  Min abstracts for one author: {min(abstracts_per_author)}")
    
    # Save the mapping to cache file
    print(f"\nSaving author-abstracts mapping to {CACHE_FILE}...")
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(author_abstracts, f)
    print(f"✓ Saved {len(author_abstracts)} author-abstract mappings to cache")

Grouping abstracts by author...


100%|██████████| 27124/27124 [00:00<00:00, 49669.74it/s]


✓ Grouped abstracts for 96176 authors

Statistics:
  Authors with abstracts: 96176
  Total abstract-author pairs: 126276
  Average abstracts per author: 1.31
  Max abstracts for one author: 98
  Min abstracts for one author: 1

Saving author-abstracts mapping to author_abstracts_cache.json...
✓ Saved 96176 author-abstract mappings to cache


## 8. Load from Cache (if available)

**Start here if you have a cache file** - you can skip cells 4, 5, 6, and 7 above.

In [13]:
# Load from cache if it exists
if os.path.exists(CACHE_FILE):
    print(f"Loading author-abstracts mapping from {CACHE_FILE}...")
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        author_abstracts = json.load(f)
    
    # We also need the authors dict for name lookups
    # If authors variable doesn't exist (cache-only path), we'll need to load it
    if 'authors' not in locals():
        print("\nNote: Loading authors from blob storage for name lookups...")
        blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
        container_client = blob_service_client.get_container_client(CONTAINER_NAME)
        
        # Quick load only authors that we have abstracts for
        authors = {}
        blob_list = container_client.list_blobs(name_starts_with=AUTHORS_PREFIX)
        for blob in tqdm(blob_list, desc="Loading author profiles"):
            blob_client = container_client.get_blob_client(blob.name)
            blob_data = blob_client.download_blob().readall()
            try:
                author_data = json.loads(blob_data)
                author_id = author_data.get('id')
                if author_id and author_id in author_abstracts:
                    authors[author_id] = author_data
            except json.JSONDecodeError:
                continue
        print(f"✓ Loaded {len(authors)} author profiles")
    
    # Show statistics
    abstracts_per_author = [len(abstracts) for abstracts in author_abstracts.values()]
    print(f"\n✓ Loaded from cache")
    print(f"  Authors with abstracts: {len(author_abstracts)}")
    print(f"  Total abstract-author pairs: {sum(abstracts_per_author)}")
    print(f"  Average abstracts per author: {sum(abstracts_per_author) / len(abstracts_per_author):.2f}")
    print(f"  Max abstracts for one author: {max(abstracts_per_author)}")
    print(f"  Min abstracts for one author: {min(abstracts_per_author)}")
else:
    print(f"✗ No cache file found at '{CACHE_FILE}'")
    print("You must run cells 4, 5, 6, and 7 above to load from Azure Blob Storage.")

Loading author-abstracts mapping from author_abstracts_cache.json...

✓ Loaded from cache
  Authors with abstracts: 96176
  Total abstract-author pairs: 126276
  Average abstracts per author: 1.31
  Max abstracts for one author: 98
  Min abstracts for one author: 1


## 9. Test Vector DB Connection

In [14]:
# Test connection to vector DB
try:
    health_response = requests.get(f"{VECTOR_DB_BASE_URL}/health", timeout=5)
    health_response.raise_for_status()
    health_data = health_response.json()
    
    print("✓ Vector DB is healthy")
    print(f"  Status: {health_data.get('status')}")
    print(f"  Milvus connected: {health_data.get('milvus_connected')}")
    print(f"  Collections: {health_data.get('collections', [])}")
except requests.exceptions.RequestException as e:
    print(f"✗ Failed to connect to Vector DB: {e}")
    print(f"  Make sure the service is running at {VECTOR_DB_BASE_URL}")
    raise

✓ Vector DB is healthy
  Status: healthy
  Milvus connected: True
  Collections: ['aegis_vectors']


## 10. Performance Profiling: Measure Embedding vs Upload Time

**Goal:** Process a sample of 100 authors to measure where time is spent:
- **Embedding computation** (CPU-bound): Can benefit from multiprocessing or GPU
- **API upload** (I/O-bound): Can benefit from threading or async

This will help us choose the right optimization strategy.

In [20]:
import time

# Load the embedding model once
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
model_load_start = time.time()
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
model_load_time = time.time() - model_load_start
print(f"✓ Model loaded in {model_load_time:.2f}s ({embedding_model.get_sentence_embedding_dimension()} dimensions)")

# Process a sample to measure timing
SAMPLE_SIZE = 100
sample_authors = list(author_abstracts.items())[:SAMPLE_SIZE]

print(f"\nProcessing {len(sample_authors)} authors to measure performance...")
print("Measuring: Embedding computation time vs API upload time\n")

results = {
    'success': 0,
    'failed': 0,
    'skipped': 0,
}

timing_stats = {
    'embedding_times': [],
    'upload_times': []
}

for author_id, abstracts in tqdm(sample_authors, desc="Profiling"):
    # Get author data
    author_data = authors.get(author_id)
    
    # Skip authors without profile data
    if not author_data or not author_data.get('name'):
        results['skipped'] += 1
        continue
    
    author_name = author_data.get('name')
    
    try:
        # TIME: Embedding computation
        embed_start = time.time()
        abstract_embeddings = embedding_model.encode(abstracts, show_progress_bar=False, convert_to_numpy=True)
        averaged_embedding = np.mean(abstract_embeddings, axis=0)
        embed_time = time.time() - embed_start
        timing_stats['embedding_times'].append(embed_time)
        
        # TIME: API upload
        upload_start = time.time()
        payload = {
            "author_id": author_id,
            "author_name": author_name,
            "embedding": averaged_embedding.tolist(),
            "num_abstracts": len(abstracts)
        }
        response = requests.post(VECTOR_UPLOAD_ENDPOINT, json=payload, timeout=30)
        response.raise_for_status()
        upload_time = time.time() - upload_start
        timing_stats['upload_times'].append(upload_time)
        
        results['success'] += 1
        
    except Exception as e:
        results['failed'] += 1
        print(f"\n✗ Failed {author_id}: {e}")

# Calculate statistics
avg_embed_time = np.mean(timing_stats['embedding_times']) if timing_stats['embedding_times'] else 0
avg_upload_time = np.mean(timing_stats['upload_times']) if timing_stats['upload_times'] else 0
total_avg_time = avg_embed_time + avg_upload_time

embed_pct = (avg_embed_time / total_avg_time * 100) if total_avg_time > 0 else 0
upload_pct = (avg_upload_time / total_avg_time * 100) if total_avg_time > 0 else 0

print(f"\n{'='*70}")
print(f"PERFORMANCE PROFILING RESULTS")
print(f"{'='*70}")
print(f"\nProcessed: {results['success']} authors successfully")
print(f"Skipped:   {results['skipped']} (no profile)")
print(f"Failed:    {results['failed']}")

print(f"\n{'='*70}")
print(f"TIMING BREAKDOWN (per author)")
print(f"{'='*70}")
print(f"  Embedding computation: {avg_embed_time:.4f}s ({embed_pct:.1f}%)")
print(f"  Upload to vector DB:   {avg_upload_time:.4f}s ({upload_pct:.1f}%)")
print(f"  Total per author:      {total_avg_time:.4f}s")

# Estimate total time for all authors
total_authors = len([a for a in author_abstracts.keys() if authors.get(a) and authors.get(a).get('name')])
estimated_sequential_hours = (total_authors * total_avg_time) / 3600

print(f"\n{'='*70}")
print(f"PROJECTIONS FOR {total_authors:,} AUTHORS")
print(f"{'='*70}")
print(f"  Sequential processing: {estimated_sequential_hours:.1f} hours")
print(f"  With 4 workers:        {estimated_sequential_hours/4:.1f} hours")
print(f"  With 8 workers:        {estimated_sequential_hours/8:.1f} hours")

# Identify bottleneck and recommend strategy
print(f"\n{'='*70}")
print(f"💡 BOTTLENECK ANALYSIS")
print(f"{'='*70}")

if avg_embed_time > avg_upload_time * 2:
    print("⚠️  BOTTLENECK: Embedding computation (CPU-bound)")
    print("\n📋 RECOMMENDATIONS:")
    print("  1. Use multiprocessing (standalone Python script, not Jupyter)")
    print("  2. Consider GPU acceleration if CUDA is available")
    print("  3. Process in batches with checkpointing for progress tracking")
elif avg_upload_time > avg_embed_time * 2:
    print("⚠️  BOTTLENECK: API upload (I/O-bound)")
    print("\n📋 RECOMMENDATIONS:")
    print("  1. Use ThreadPoolExecutor (works in Jupyter)")
    print("  2. Increase concurrent uploads (start with 4-8 threads)")
    print("  3. Consider async/await with aiohttp for better concurrency")
    print("  4. Batch API requests if the endpoint supports it")
else:
    print("⚙️  BALANCED: Both operations take similar time")
    print("\n📋 RECOMMENDATIONS:")
    print("  1. Use threading for moderate speedup (simpler, stable in Jupyter)")
    print("  2. Start with 4 threads and monitor performance")
    print("  3. Consider moving to standalone script if >4x speedup needed")

print(f"\n{'='*70}\n")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...
✓ Model loaded in 0.70s (384 dimensions)

Processing 100 authors to measure performance...
Measuring: Embedding computation time vs API upload time



Profiling: 100%|██████████| 100/100 [00:09<00:00, 11.10it/s]


PERFORMANCE PROFILING RESULTS

Processed: 100 authors successfully
Skipped:   0 (no profile)
Failed:    0

TIMING BREAKDOWN (per author)
  Embedding computation: 0.0714s (79.8%)
  Upload to vector DB:   0.0181s (20.2%)
  Total per author:      0.0895s

PROJECTIONS FOR 93,007 AUTHORS
  Sequential processing: 2.3 hours
  With 4 workers:        0.6 hours
  With 8 workers:        0.3 hours

💡 BOTTLENECK ANALYSIS
⚠️  BOTTLENECK: Embedding computation (CPU-bound)

📋 RECOMMENDATIONS:
  1. Use multiprocessing (standalone Python script, not Jupyter)
  2. Consider GPU acceleration if CUDA is available
  3. Process in batches with checkpointing for progress tracking




## 11. Full Upload: Process All Authors

**Important:** This will process all ~90,000 authors sequentially. Based on your profiling results above, this will take the estimated time shown in the projections.

Run this cell to upload all author embeddings to the vector database.

In [21]:
import time

# Reuse embedding model if already loaded, otherwise load it
if 'embedding_model' not in locals() or embedding_model is None:
    print(f"Loading embedding model: {EMBEDDING_MODEL}...")
    model_load_start = time.time()
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    model_load_time = time.time() - model_load_start
    print(f"✓ Model loaded in {model_load_time:.2f}s ({embedding_model.get_sentence_embedding_dimension()} dimensions)")
else:
    print(f"✓ Reusing existing embedding model ({embedding_model.get_sentence_embedding_dimension()} dimensions)")

# Filter to authors with profiles
authors_to_process = [(author_id, abstracts) for author_id, abstracts in author_abstracts.items() 
                      if authors.get(author_id) and authors.get(author_id).get('name')]

print(f"\nProcessing {len(authors_to_process):,} authors with profiles...")
print(f"Starting full upload at {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

results = {
    'success': 0,
    'failed': 0,
    'skipped': 0,
}

failed_authors = []
start_time = time.time()

for author_id, abstracts in tqdm(authors_to_process, desc="Uploading"):
    author_data = authors.get(author_id)
    
    # Double-check author has profile data
    if not author_data or not author_data.get('name'):
        results['skipped'] += 1
        continue
    
    author_name = author_data.get('name')
    
    try:
        # Compute embedding
        abstract_embeddings = embedding_model.encode(abstracts, show_progress_bar=False, convert_to_numpy=True)
        averaged_embedding = np.mean(abstract_embeddings, axis=0)
        
        # Upload to vector DB
        payload = {
            "author_id": author_id,
            "author_name": author_name,
            "embedding": averaged_embedding.tolist(),
            "num_abstracts": len(abstracts)
        }
        response = requests.post(VECTOR_UPLOAD_ENDPOINT, json=payload, timeout=30)
        response.raise_for_status()
        
        results['success'] += 1
        
    except Exception as e:
        results['failed'] += 1
        failed_authors.append((author_id, str(e)))
        # Only print first 10 errors to avoid cluttering output
        if results['failed'] <= 10:
            print(f"\n✗ Failed {author_id}: {e}")
        elif results['failed'] == 11:
            print(f"\n  (Suppressing further error messages...)")

# Final statistics
elapsed_time = time.time() - start_time
print(f"\n{'='*70}")
print(f"UPLOAD COMPLETE!")
print(f"{'='*70}")
print(f"Finished at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time: {elapsed_time/3600:.2f} hours ({elapsed_time/60:.1f} minutes)")
print(f"\n✓ Successful uploads: {results['success']:,}")
print(f"⊘ Skipped (no profile): {results['skipped']}")
print(f"✗ Failed uploads: {results['failed']}")

if results['success'] > 0:
    print(f"\nAverage rate: {results['success']/elapsed_time:.2f} authors/second")

if results['failed'] > 0:
    print(f"\n⚠️  {results['failed']} authors failed to upload")
    print(f"First few failures saved in 'failed_authors' variable for inspection")

print(f"{'='*70}\n")

✓ Reusing existing embedding model (384 dimensions)

Processing 93,007 authors with profiles...
Starting full upload at 2026-02-28 01:42:28



Uploading:   5%|▌         | 5004/93007 [04:29<1:11:46, 20.43it/s]


  Progress: 5,000/93,007 authors
  Rate: 18.6 authors/sec
  Elapsed: 0.1h, Remaining: ~1.3h


Uploading:  11%|█         | 10005/93007 [08:27<54:54, 25.20it/s] 


  Progress: 10,000/93,007 authors
  Rate: 19.7 authors/sec
  Elapsed: 0.1h, Remaining: ~1.2h


...


  Progress: 85,000/93,007 authors
  Rate: 25.6 authors/sec
  Elapsed: 0.9h, Remaining: ~0.1h


Uploading:  97%|█████████▋| 90005/93007 [58:26<01:53, 26.48it/s]


  Progress: 90,000/93,007 authors
  Rate: 25.7 authors/sec
  Elapsed: 1.0h, Remaining: ~0.0h


Uploading: 100%|██████████| 93007/93007 [1:00:13<00:00, 25.74it/s]


UPLOAD COMPLETE!
Finished at: 2026-02-28 02:42:42
Total time: 1.00 hours (60.2 minutes)

✓ Successful uploads: 93,007
⊘ Skipped (no profile): 0
✗ Failed uploads: 0

Average rate: 25.74 authors/second



## 12. Flush Collection to Make Data Searchable

**Important:** After uploading data, we need to flush the collection to make the data visible to search queries. This persists the data and updates the search index.

In [28]:
# Flush the collection to make uploaded data searchable
from pymilvus import Collection, connections, utility

try:
    # Connect to Milvus
    print("Connecting to Milvus...")
    connections.connect(
        alias="default",
        host="localhost",
        port="19530"
    )
    print("✓ Connected to Milvus")
    
    # Check what collections exist
    collections = utility.list_collections()
    print(f"Available collections: {collections}")
    
    collection_name = 'aegis_vectors'  # Collection name used by the vector-db service
    
    # Check if collection exists
    if not utility.has_collection(collection_name):
        print(f"\n✗ Collection '{collection_name}' does not exist!")
        print(f"\nPossible reasons:")
        print(f"  1. No data has been uploaded yet - run the upload cells first")
        print(f"  2. The vector-db service needs to be restarted to create the collection")
        print(f"\nTo fix:")
        print(f"  - Run cell 22 (Performance Profiling) or cell 24 (Full Upload) first")
        print(f"  - The collection will be created automatically on first upload")
    else:
        collection = Collection(collection_name)
        
        print(f"\nFlushing collection '{collection_name}' to persist data...")
        print(f"Entities before flush: {collection.num_entities}")
        
        # Flush the collection
        collection.flush()
        
        print(f"✓ Collection flushed successfully!")
        print(f"Total entities in collection: {collection.num_entities}")
        
        # Ensure collection is loaded for searching
        collection.load()
        print(f"✓ Collection loaded and ready for searches")
    
except Exception as e:
    print(f"✗ Error: {e}")

Connecting to Milvus...
✓ Connected to Milvus
Available collections: ['aegis_vectors']

Flushing collection 'aegis_vectors' to persist data...
Entities before flush: 89075
✓ Collection flushed successfully!
Total entities in collection: 93007
✓ Collection loaded and ready for searches


## 13. Debug: Inspect Search Results Structure

In [32]:
# Debug: Test a single search and inspect the raw response structure
test_query = "machine learning"
search_endpoint = f"{VECTOR_DB_BASE_URL}/search/text"
payload = {
    "query_text": test_query,
    "limit": 3,
    "collection_name": "aegis_vectors"
}

print(f"Testing search with query: '{test_query}'")
print(f"Endpoint: {search_endpoint}")
print(f"Payload: {payload}\n")

try:
    response = requests.post(search_endpoint, json=payload, timeout=30)
    response.raise_for_status()
    results = response.json()
    
    print("Raw response structure:")
    print(json.dumps(results, indent=2)[:2000])  # First 2000 chars
    
    if results.get('results'):
        print(f"\nFirst result entity keys: {results['results'][0].get('entity', {}).keys()}")
        print(f"First result entity: {results['results'][0].get('entity', {})}")
    
except Exception as e:
    print(f"✗ Search failed: {e}")
    import traceback
    traceback.print_exc()

Testing search with query: 'machine learning'
Endpoint: http://localhost:8002/search/text
Payload: {'query_text': 'machine learning', 'limit': 3, 'collection_name': 'aegis_vectors'}

Raw response structure:
{
  "results": [
    {
      "id": "author_author_138157d0-5841-5467-a7c8-4574b77ab35b",
      "distance": 0.7523834705352783,
      "entity": {
        "id": "author_author_138157d0-5841-5467-a7c8-4574b77ab35b",
        "distance": 0.7523834705352783,
        "entity": {
          "id": "author_author_138157d0-5841-5467-a7c8-4574b77ab35b",
          "author_id": "author_138157d0-5841-5467-a7c8-4574b77ab35b",
          "author_name": "Jaime G. Carbonell",
          "num_abstracts": 1,
          "embedding": [
            0.05319513380527496,
            -0.0933387503027916,
            0.01739821583032608,
            -0.034286294132471085,
            -0.018109215423464775,
            -0.02945774979889393,
            0.021140849217772484,
            -0.04703538492321968,
       

## 14. Test Search Functionality

In [45]:
# Test text-based search
def search_authors(query_text: str, limit: int = 5) -> dict:
    """
    Search for authors using text query.
    
    Args:
        query_text: Text query to search for
        limit: Maximum number of results
        
    Returns:
        Search results dictionary
    """
    search_endpoint = f"{VECTOR_DB_BASE_URL}/search/text"
    payload = {
        "query_text": query_text,
        "limit": limit,
        "collection_name": "aegis_vectors"  # Specify the collection to search
    }
    
    response = requests.post(search_endpoint, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

# Example search queries
test_queries = [
    "machine learning and artificial intelligence",
    "network security and cybersecurity",
    "quantum computing research"
]

print("Testing search functionality with sample queries:\n")
for query in test_queries:
    print(f"Query: '{query}'")
    print("-" * 60)
    
    try:
        results = search_authors(query, limit=5)
        
        print(f"Search time: {results.get('search_time_ms', 0):.2f}ms")
        print(f"Found {len(results.get('results', []))} results:\n")
        
        for i, result in enumerate(results.get('results', []), 1):
            # Entity data is now properly flattened in the API
            entity = result.get('entity', {})
            
            print(f"  {i}. {entity.get('author_name', 'Unknown')}")
            print(f"     Author ID: {entity.get('author_id', 'N/A')}")
            print(f"     Distance: {result.get('distance', 0):.4f}")
            print(f"     Abstracts: {entity.get('num_abstracts', 0)}")
            print()
    
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Search failed: {e}\n")
    
    print()

Testing search functionality with sample queries:

Query: 'machine learning and artificial intelligence'
------------------------------------------------------------
Search time: 8.37ms
Found 5 results:

  1. Pat Langley
     Author ID: author_e7e94a99-a4fe-5225-bf12-7143c895a892
     Distance: 0.7682
     Abstracts: 4

  2. Jaime G. Carbonell
     Author ID: author_138157d0-5841-5467-a7c8-4574b77ab35b
     Distance: 0.8270
     Abstracts: 1

  3. Gail A. Carpenter
     Author ID: author_08c22544-e117-597a-a340-1853cb28d7c1
     Distance: 0.8762
     Abstracts: 4

  4. Jude W. Shavlik
     Author ID: author_80c206c3-87cc-55e6-b0af-4037dde9a91d
     Distance: 0.8836
     Abstracts: 4

  5. Justin Zhan
     Author ID: author_d1fdcbd1-2161-50af-b390-acbf1e7464ad
     Distance: 0.9048
     Abstracts: 4


Query: 'network security and cybersecurity'
------------------------------------------------------------
Search time: 6.31ms
Found 5 results:

  1. Sushil Jajodia
     Author ID: author_56